In [ ]:
from pyspark.sql import functions as F

# tạo bảng gold
df_silver = spark.read.table("silver_crypto_prices")

df_gold = df_silver.withColumn("symbol", F.regexp_replace("symbol", "USDT", ""))\
                        .withColumn("time_minute", F.date_trunc("minute",F.col("timestamp")))\
                        .groupBy("symbol", "time_minute", "category")\
                        .agg(
                            F.round(F.avg("price"), 8).alias("price_avg_usdt"),
                            F.round(F.max("price"), 8).alias("price_max_usdt"),
                            F.round(F.min("price"), 8).alias("price_min_usdt"))\
                        .select('symbol','price_avg_usdt','price_min_usdt','price_max_usdt','time_minute','category')\
                        .orderBy("time_minute", "symbol")

df_gold.write.mode("overwrite").format("delta").saveAsTable("gold_crypto_analysis")

# tạo bảng dim_date
dim_time = df_gold.select("time_minute").distinct() \
                        .withColumn("hour", F.hour("time_minute")) \
                        .withColumn("day_of_week", F.date_format("time_minute", "EEEE")) \
                        .withColumn("is_weekend", F.when(F.dayofweek("time_minute").isin(1, 7), "Weekend").otherwise("Weekday"))

dim_time.write.mode("overwrite").format("delta").saveAsTable("dim_time")

In [ ]:
spark.sql("OPTIMIZE bronze_crypto_prices")
spark.sql("OPTIMIZE silver_crypto_prices")
spark.sql("OPTIMIZE gold_crypto_analysis")